# Equity Alpha Engine: research walkthrough

A guided tour of the pipeline on real market data. Each section mirrors one module,
so you can see what the code produces at every stage rather than only the final numbers.

**Prerequisites**
```bash
pip install -e ".[all]"
python scripts/fetch_data.py --config configs/default.yaml
```

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from alpha_engine.config import load_config
from alpha_engine.data.loader import load_market_data
from alpha_engine.features.pipeline import build_feature_set
from alpha_engine.validation.splitters import PurgedWalkForward
from alpha_engine.models.trainer import walk_forward_train
from alpha_engine.evaluation.metrics import daily_ic, ic_summary, quantile_spread
from alpha_engine.reporting.figures import apply_style

apply_style()
cfg = load_config("../configs/default.yaml")
cfg.data.cache_dir = "../data/cache"
print(f"universe={cfg.data.universe}  horizon={cfg.label.horizon}d  folds={cfg.validation.n_splits}")

## 1. Data

The universe is reconstructed point in time, so past dates carry the members the index
actually held rather than the ones that happened to survive. Check the coverage figure:
it is the honest measure of how much survivorship bias remains.

In [ ]:
md = load_market_data(cfg)
summary = md.summary()
for k, v in summary.items():
    if k != "download":
        print(f"{k:34s} {v}")
dl = summary.get("download", {})
print(f"\npoint-in-time coverage: {100*dl.get('coverage',0):.1f}% "
      f"({dl.get('n_succeeded')} of {dl.get('n_requested')} historical members priced)")

## 2. Features and the label

36 factors, each winsorised and z-scored **within each date**. The normalisation is what
makes a volatility reading in March 2020 comparable with one from 2017.

In [ ]:
fs = build_feature_set(md, cfg)
print(fs.summary())
fs.frame[["date", "ticker", "target", "mom_12_1", "vol_63", "amihud_illiq"]].tail()

In [ ]:
# Univariate signal check: is any single factor doing anything?
rows = []
for f in ["mom_12_1", "ret_21", "vol_63", "amihud_illiq", "log_dollar_volume", "rsi_14"]:
    ic = fs.frame.groupby("date", observed=True).apply(
        lambda g: g[f].corr(g["target"], method="spearman"), include_groups=False).dropna()
    rows.append({"feature": f, **ic_summary(ic, cfg.label.horizon)})
pd.DataFrame(rows)[["feature", "mean_ic", "icir", "t_stat_nw", "hit_rate"]].round(4)

## 3. Why ordinary cross-validation fails here

A 21-day label formed on the first of the month is still being realised at month end.
Purging removes training rows whose label window reaches into the test window; the embargo
adds a further buffer. Confirm the gap is at least the label horizon.

In [ ]:
cv = PurgedWalkForward(
    n_splits=cfg.validation.n_splits,
    train_window_days=cfg.validation.train_window_days,
    test_window_days=cfg.validation.test_window_days,
    label_horizon=cfg.label.horizon,
    embargo_days=cfg.validation.embargo_days,
)
folds = cv.get_folds(fs.frame["date"])
pd.DataFrame([f.describe() for f in folds])

## 4. Walk-forward training

Every model is fitted on the past and scored on the future, fold by fold. Concatenating the
test-window predictions gives one continuous out-of-sample track record per model.

In [ ]:
cfg.models.enabled = ["ridge", "lightgbm", "ensemble"]   # drop the GRU for a faster notebook
result = walk_forward_train(fs, cfg)
result.overall_metrics[["model", "rank_mean_ic", "rank_icir", "rank_t_stat_nw", "q5_minus_q1"]].round(4)

In [ ]:
# Fold-by-fold stability matters more than the average.
piv = result.fold_metrics.pivot_table(index="fold", columns="model", values="rank_mean_ic")
ax = piv.plot(marker="o", figsize=(9, 4))
ax.axhline(0, color="grey", lw=1)
ax.set_title("Information coefficient by walk-forward fold")
ax.set_ylabel("mean IC"); plt.show()

## 5. From forecast to portfolio

Note the beta neutralisation. Equal dollars long and short is **not** market neutral, and
skipping this step is what turned a positive information coefficient into a 28% annual loss
in an early version of this engine.

In [ ]:
from alpha_engine.backtest.engine import run_backtest
from alpha_engine.evaluation.performance import performance_stats

best = result.overall_metrics.iloc[0]["model"]
bt = run_backtest(result.predictions, f"pred_{best}", cfg.portfolio, cfg.backtest,
                  label_horizon=cfg.label.horizon)
pd.Series(performance_stats(bt.returns)).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bt.equity_curve.plot(ax=ax)
ax.axhline(1.0, color="grey", lw=1)
ax.set_title(f"Out-of-sample equity curve, net of costs: {best}")
ax.set_ylabel("growth of 1"); plt.show()

print(f"mean net beta per rebalance: {bt.exposure['net_beta'].mean():+.4f}")

## 6. Is it real?

The deflated Sharpe ratio asks the only question that matters after a search: given that
this was the best of N attempts, what is the probability the true Sharpe is above the
selection-adjusted threshold?

In [ ]:
from alpha_engine.evaluation.significance import (
    deflated_sharpe_ratio, stationary_bootstrap_sharpe, probability_of_backtest_overfitting)

print(pd.Series(deflated_sharpe_ratio(bt.returns, n_trials=60)).round(4).to_string())
print()
print(pd.Series(stationary_bootstrap_sharpe(bt.returns, n_boot=500)).round(4).to_string())

## 7. The leakage control

Real markets never tell you what the true predictable component was. The simulator does,
and it can set it to exactly zero. Any model reporting a reliable information coefficient
on null data has found a bug rather than a signal.

In [ ]:
null_cfg = load_config("../configs/leakage_control.yaml")
null_cfg.data.cache_dir = "../data/cache"
null_cfg.data.n_assets = 60
null_cfg.validation.n_splits = 2
null_cfg.models.enabled = ["ridge", "lightgbm"]

null_md = load_market_data(null_cfg, cache=False)
null_fs = build_feature_set(null_md, null_cfg)
null_res = walk_forward_train(null_fs, null_cfg)
null_res.overall_metrics[["model", "rank_mean_ic", "rank_t_stat_nw"]].round(4)

Both t-statistics should sit comfortably inside plus or minus 2. If they do not, something
in the pipeline is reading the future.

---

Full design notes, including the two targets that were tested and rejected, are in
[`docs/METHODOLOGY.md`](../docs/METHODOLOGY.md).